# Imports

In [ ]:
!pip install tensorflow pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_curve, auc, roc_auc_score)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.nn.functional as F

import warnings
warnings.filterwarnings('ignore')

In [ ]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Load data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')



In [ ]:
# Update path to your file location
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Loan_Default.csv')
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")

# Data preprocessing

In [ ]:
# Create a copy for preprocessing
df_processed = df.copy()

# Handle missing values
print("Handling missing values...")
numerical_cols = df_processed.select_dtypes(include=[np.number]).columns
for col in numerical_cols:
    df_processed[col].fillna(df_processed[col].median(), inplace=True)

categorical_cols = df_processed.select_dtypes(include=['object']).columns
for col in categorical_cols:
    df_processed[col].fillna(df_processed[col].mode(), inplace=True)

# Encode categorical variables
label_encoders = {}

if 'loan_type' in df_processed.columns:
    le_loan_type = LabelEncoder()
    df_processed['loan_type_encoded'] = le_loan_type.fit_transform(df_processed['loan_type'])
    label_encoders['loan_type'] = le_loan_type

if 'loan_purpose' in df_processed.columns:
    le_loan_purpose = LabelEncoder()
    df_processed['loan_purpose_encoded'] = le_loan_purpose.fit_transform(df_processed['loan_purpose'])
    label_encoders['loan_purpose'] = le_loan_purpose

if 'Region' in df_processed.columns:
    le_region = LabelEncoder()
    df_processed['Region_encoded'] = le_region.fit_transform(df_processed['Region'])
    label_encoders['Region'] = le_region

# Encode target variable
target_col = 'Status' if 'Status' in df_processed.columns else 'status' if 'status' in df_processed.columns else 'loan_status'
le_target = LabelEncoder()
df_processed['target'] = le_target.fit_transform(df_processed[target_col])
label_encoders['target'] = le_target

# Handle age column
if 'age' in df_processed.columns:
    if df_processed['age'].dtype == 'object':
        age_mapping = {
            '<25': 22, '25-34': 29.5, '35-44': 39.5,
            '45-54': 49.5, '55-64': 59.5, '65-74': 69.5, '>74': 77
        }
        df_processed['age_numeric'] = df_processed['age'].map(age_mapping)
        df_processed['age_numeric'].fillna(df_processed['age_numeric'].median(), inplace=True)
    else:
        df_processed['age_numeric'] = df_processed['age']

# Define feature columns
feature_columns = [
    'income', 'Credit_Score', 'loan_amount', 'dtir1', 'LTV',
    'age_numeric', 'open_credit', 'property_value', 'rate_of_interest',
    'loan_type_encoded', 'loan_purpose_encoded', 'Region_encoded'
]

feature_columns = [col for col in feature_columns if col in df_processed.columns]

# Prepare X and y
X = df_processed[feature_columns].values
y = df_processed['target'].values

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print(f"Features used: {feature_columns}")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Convert to DataFrame
X_train_df = pd.DataFrame(X_train, columns=feature_columns)
X_test_df = pd.DataFrame(X_test, columns=feature_columns)

print("Step 1: Converting all columns to numeric...")
# Convert all columns to numeric, coercing errors to NaN
for col in X_train_df.columns:
    X_train_df[col] = pd.to_numeric(X_train_df[col], errors='coerce')
    X_test_df[col] = pd.to_numeric(X_test_df[col], errors='coerce')

print("  ✅ Converted to numeric")

print("\nStep 2: Filling NaN values with median...")
# Fill NaN with median for each column
for col in X_train_df.columns:
    nan_before = X_train_df[col].isna().sum()
    if nan_before > 0:
        median_val = X_train_df[col].median()
        X_train_df[col].fillna(median_val, inplace=True)
        X_test_df[col].fillna(median_val, inplace=True)
        print(f"  Filled {col}: {nan_before} NaN values → median={median_val:.2f}")

print("  ✅ All NaN values filled")

# Convert back to numpy arrays
X_train = X_train_df.values.astype(np.float32)
X_test = X_test_df.values.astype(np.float32)

print("\nStep 3: Verifying data...")
# Verify no NaN/Inf values remain
nan_count = np.isnan(X_train).sum()
inf_count = np.isinf(X_train).sum()
print(f"  NaN values: {nan_count}")
print(f"  Inf values: {inf_count}")

if nan_count > 0 or inf_count > 0:
    print(f"  ❌ Still have issues - filling remaining NaN with 0...")
    X_train = np.nan_to_num(X_train, nan=0.0, posinf=1e6, neginf=-1e6)
    X_test = np.nan_to_num(X_test, nan=0.0, posinf=1e6, neginf=-1e6)
    print(f"  ✅ Fixed")

print("\nStep 4: Normalizing data...")
# Normalize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"  X_train_scaled: {X_train_scaled.shape}")
print(f"  X_test_scaled: {X_test_scaled.shape}")

print("\nStep 5: Final data quality check...")
nan_count = np.isnan(X_train_scaled).sum()
inf_count = np.isinf(X_train_scaled).sum()
print(f"  NaN count: {nan_count}")
print(f"  Inf count: {inf_count}")
print(f"  Data range: [{X_train_scaled.min():.4f}, {X_train_scaled.max():.4f}]")

if nan_count == 0 and inf_count == 0:
    print("\n✅ Data is clean and ready!")
else:
    print("\n❌ Data still has issues")

print("\nStep 6: Converting to PyTorch tensors...")
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

X_train_tensor = torch.FloatTensor(X_train_scaled).to(device)
y_train_tensor = torch.FloatTensor(y_train).reshape(-1, 1).to(device)
X_test_tensor = torch.FloatTensor(X_test_scaled).to(device)
y_test_tensor = torch.FloatTensor(y_test).reshape(-1, 1).to(device)

print(f"  X_train_tensor: {X_train_tensor.shape}")
print(f"  y_train_tensor: {y_train_tensor.shape}")
print(f"  Device: {device}")

print("\n" + "="*60)
print("✅ PREPROCESSING COMPLETE - Ready for training!")
print("="*60)


# Spilit and normalize train - val data

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Convert to DataFrame temporarily for easier handling
X_train_df = pd.DataFrame(X_train, columns=feature_columns)
X_test_df = pd.DataFrame(X_test, columns=feature_columns)

print("Step 1: Filling NaN values...")
# Fill NaN with median for each column
for col in X_train_df.columns:
    if X_train_df[col].isna().sum() > 0:
        median_val = X_train_df[col].median()
        X_train_df[col].fillna(median_val, inplace=True)
        X_test_df[col].fillna(median_val, inplace=True)
        print(f"  Filled {col}")

# Convert back to numpy arrays
X_train = X_train_df.values
X_test = X_test_df.values

print("\nStep 2: Checking for remaining issues...")
# Verify no NaN values remain
nan_count = np.isnan(X_train).sum()
print(f"  NaN values in X_train: {nan_count}")

if nan_count == 0:
    print("  ✅ All NaN values filled!")
else:
    print(f"  ❌ Still have {nan_count} NaN values")

print("\nStep 3: Normalizing data...")
# Normalize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")

print("\nStep 4: Final verification...")
nan_count = np.isnan(X_train_scaled).sum()
inf_count = np.isinf(X_train_scaled).sum()
print(f"  NaN count: {nan_count}")
print(f"  Inf count: {inf_count}")

if nan_count == 0 and inf_count == 0:
    print("\n✅ Data is clean and ready for training!")
else:
    print("\n❌ Data still has issues")

print("\nStep 5: Converting to PyTorch tensors...")
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

X_train_tensor = torch.FloatTensor(X_train_scaled).to(device)
y_train_tensor = torch.FloatTensor(y_train).reshape(-1, 1).to(device)
X_test_tensor = torch.FloatTensor(X_test_scaled).to(device)
y_test_tensor = torch.FloatTensor(y_test).reshape(-1, 1).to(device)

print(f"  X_train_tensor: {X_train_tensor.shape}")
print(f"  y_train_tensor: {y_train_tensor.shape}")
print(f"  X_test_tensor: {X_test_tensor.shape}")
print(f"  y_test_tensor: {y_test_tensor.shape}")
print(f"  Device: {device}")

print("\n✅ All preprocessing complete!")


In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# Check and remove columns with >50% missing data
columns_to_remove = []

print("Checking for columns with >50% missing data...")
for i in range(X_train.shape[1]):  # ← FIX: use X_train.shape[1] for number of columns
    nan_pct = (np.isnan(X_train[:, i]).sum() / len(X_train)) * 100
    if nan_pct > 50:
        columns_to_remove.append(i)
        print(f"  Removing column {i} - {nan_pct:.1f}% missing")

# Remove problematic columns
if len(columns_to_remove) > 0:
    X_train = np.delete(X_train, columns_to_remove, axis=1)
    X_test = np.delete(X_test, columns_to_remove, axis=1)
    print(f"Removed {len(columns_to_remove)} columns")

print(f"\nAfter cleaning:")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

# Normalize features
print("\nNormalizing data...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Verify data quality
print("\nData quality verification:")
nan_count = np.isnan(X_train_scaled).sum()
inf_count = np.isinf(X_train_scaled).sum()
print(f"  NaN count: {nan_count}")
print(f"  Inf count: {inf_count}")

if nan_count == 0 and inf_count == 0:
    print("\n✅ Data is clean and ready for training!")
else:
    print("\n❌ Data still has issues. Check preprocessing.")

# Convert to PyTorch tensors
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

X_train_tensor = torch.FloatTensor(X_train_scaled).to(device)
y_train_tensor = torch.FloatTensor(y_train).to(device)
X_test_tensor = torch.FloatTensor(X_test_scaled).to(device)
y_test_tensor = torch.FloatTensor(y_test).to(device)

print(f"\nTensor shapes:")
print(f"  X_train: {X_train_tensor.shape}")
print(f"  y_train: {y_train_tensor.shape}")
print(f"  X_test: {X_test_tensor.shape}")
print(f"  y_test: {y_test_tensor.shape}")
print(f"  Device: {device}")


## Data loader

In [ ]:
# Create TensorDataset
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Create DataLoaders
batch_size = 64  # Increased for more stable training
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")


# Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CreditRiskClassifier(nn.Module):
    """Optimized Neural Network for Credit Risk Prediction"""

    def __init__(self, input_dim, hidden_dims=[128, 64, 32], dropout_rates=[0.4, 0.3, 0.2]):
        super(CreditRiskClassifier, self).__init__()

        # Ensure all dimensions are integers
        input_dim = int(input_dim)
        hidden_dims = [int(h) for h in hidden_dims]

        # Input layer
        self.fc1 = nn.Linear(input_dim, hidden_dims[0])
        self.bn1 = nn.BatchNorm1d(hidden_dims[0])
        self.dropout1 = nn.Dropout(dropout_rates[0])

        # Hidden layer 2
        self.fc2 = nn.Linear(hidden_dims[0], hidden_dims[1])
        self.bn2 = nn.BatchNorm1d(hidden_dims[1])
        self.dropout2 = nn.Dropout(dropout_rates[1])

        # Hidden layer 3
        self.fc3 = nn.Linear(hidden_dims[1], hidden_dims[2])
        self.bn3 = nn.BatchNorm1d(hidden_dims[2])
        self.dropout3 = nn.Dropout(dropout_rates[2])

        # Output layer
        self.fc4 = nn.Linear(hidden_dims[2], 1)

        # Initialize weights
        self._init_weights()

    def _init_weights(self):
        """Initialize weights using He initialization"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        # Layer 1
        x = self.fc1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.dropout1(x)

        # Layer 2
        x = self.fc2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.dropout2(x)

        # Layer 3
        x = self.fc3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.dropout3(x)

        # Output layer
        x = self.fc4(x)
        x = torch.sigmoid(x)

        return x

# Now initialize the model
input_dim = int(X_train_scaled.shape[1])

print(f"Creating model with input_dim={input_dim}")
print(f"Input type: {type(input_dim)}")

model = CreditRiskClassifier(
    input_dim=input_dim,
    hidden_dims=[128, 64, 32],
    dropout_rates=[0.4, 0.3, 0.2]
).to(device)

print("="*60)
print("MODEL ARCHITECTURE")
print("="*60)
print(model)
print(f"\nInput dimension: {input_dim}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("="*60)


# Train model

In [ ]:
# Loss function with class weights (handle class imbalance)
pos_weight = torch.tensor([len(y_train[y_train==0]) / len(y_train[y_train==1])]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Optimizer - AdamW with weight decay
optimizer = optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4,  # L2 regularization
    betas=(0.9, 0.999)
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

# Early stopping
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

early_stopping = EarlyStopping(patience=15, min_delta=0.001)

print("✅ Training setup complete")


In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        # Forward pass
        optimizer.zero_grad()
        outputs = model(batch_X).squeeze()
        loss = criterion(outputs, batch_y)

        # Backward pass
        loss.backward()

        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        # Statistics
        running_loss += loss.item()
        predicted = (outputs > 0.5).float()
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def validate_epoch(model, test_loader, criterion, device):
    """Validate for one epoch"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)

            outputs = model(batch_X).squeeze()
            loss = criterion(outputs, batch_y)

            running_loss += loss.item()
            predicted = (outputs > 0.5).float()
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())

    epoch_loss = running_loss / len(test_loader)
    epoch_acc = correct / total

    # Calculate additional metrics
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)

    return epoch_loss, epoch_acc, precision, recall, f1


In [ ]:
# Training configuration
num_epochs = 100

# Storage for history
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [],
    'val_precision': [], 'val_recall': [], 'val_f1': []
}

print("="*60)
print("STARTING MODEL TRAINING")
print("="*60)

best_val_loss = float('inf')
best_model_state = None

for epoch in range(num_epochs):
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)

    # Validate
    val_loss, val_acc, val_precision, val_recall, val_f1 = validate_epoch(
        model, test_loader, criterion, device
    )

    # Store history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_precision'].append(val_precision)
    history['val_recall'].append(val_recall)
    history['val_f1'].append(val_f1)

    # Update learning rate
    scheduler.step(val_loss)

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict().copy()

    # Print progress every 5 epochs
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}]")
        print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"  Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        print(f"  Val Precision: {val_precision:.4f} | Val Recall: {val_recall:.4f} | Val F1: {val_f1:.4f}")
        print(f"  LR: {optimizer.param_groups[0]['lr']:.6f}")
        print("-" * 60)

    # Early stopping
    early_stopping(val_loss)
    if early_stopping.early_stop:
        print(f"\n⚠️ Early stopping triggered at epoch {epoch+1}")
        break

# Load best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\n✅ Loaded best model with validation loss: {best_val_loss:.4f}")

print("\n✅ Training completed!")


# Visualize training

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Loss plot
axes[0, 0].plot(history['train_loss'], label='Training Loss', linewidth=2, color='#3b82f6')
axes[0, 0].plot(history['val_loss'], label='Validation Loss', linewidth=2, color='#ef4444')
axes[0, 0].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Accuracy plot
axes[0, 1].plot(history['train_acc'], label='Training Accuracy', linewidth=2, color='#10b981')
axes[0, 1].plot(history['val_acc'], label='Validation Accuracy', linewidth=2, color='#f59e0b')
axes[0, 1].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Precision plot
axes[1, 0].plot(history['val_precision'], label='Validation Precision', linewidth=2, color='#8b5cf6')
axes[1, 0].set_title('Model Precision', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Recall & F1 plot
axes[1, 1].plot(history['val_recall'], label='Validation Recall', linewidth=2, color='#ec4899')
axes[1, 1].plot(history['val_f1'], label='Validation F1', linewidth=2, color='#06b6d4')
axes[1, 1].set_title('Model Recall & F1', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Score')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


# Evaluate model

In [ ]:
# Get predictions
model.eval()
with torch.no_grad():
    y_pred_proba = model(X_test_tensor).cpu().numpy().flatten()
    y_pred = (y_pred_proba > 0.5).astype(int)

y_test_np = y_test

# Calculate metrics
accuracy = accuracy_score(y_test_np, y_pred)
precision = precision_score(y_test_np, y_pred)
recall = recall_score(y_test_np, y_pred)
f1 = f1_score(y_test_np, y_pred)
roc_auc = roc_auc_score(y_test_np, y_pred_proba)

print("="*60)
print("MODEL EVALUATION METRICS")
print("="*60)
print(f"Accuracy:  {accuracy*100:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall:    {recall*100:.2f}%")
print(f"F1 Score:  {f1*100:.2f}%")
print(f"ROC AUC:   {roc_auc*100:.2f}%")
print("="*60)

# Confusion Matrix
cm = confusion_matrix(y_test_np, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=['Predicted Bad', 'Predicted Good'],
            yticklabels=['Actual Bad', 'Actual Good'])
plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test_np, y_pred_proba)

plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, color='#2563eb', linewidth=2, label=f'ROC Curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='#ef4444', linestyle='--', linewidth=2, label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# Save model

In [ ]:
# Save complete model
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scaler': scaler,
    'label_encoders': label_encoders,
    'feature_columns': feature_columns,
    'best_val_loss': best_val_loss,
    'history': history
}, 'credit_risk_pytorch_model.pth')

print("✅ Model saved as 'credit_risk_pytorch_model.pth'")

# Download model
from google.colab import files
files.download('credit_risk_pytorch_model.pth')
print("📥 Model downloaded")


# Make prediction

In [ ]:
def predict_loan_application(model, application_data, scaler, label_encoders, feature_columns, device):
    """Predict credit risk for a new loan application"""
    model.eval()

    # Create feature vector
    features = []
    for col in feature_columns:
        if col.endswith('_encoded'):
            original_col = col.replace('_encoded', '')
            if original_col in application_data and original_col in label_encoders:
                try:
                    # Try to encode the value
                    encoded_value = label_encoders[original_col].transform([application_data[original_col]])[0]
                    features.append(encoded_value)
                except ValueError:
                    # If value not seen in training, use the most common class (mode)
                    print(f"⚠️ Warning: '{application_data[original_col]}' not seen in training for {original_col}")
                    print(f"   Using default value 0")
                    features.append(0)
            else:
                features.append(0)
        elif col == 'age_numeric':
            features.append(application_data.get('age_numeric', application_data.get('age', 35)))
        else:
            features.append(application_data.get(col, 0))

    # Scale and predict
    features_array = np.array(features, dtype=np.float32).reshape(1, -1)
    features_scaled = scaler.transform(features_array)
    features_tensor = torch.FloatTensor(features_scaled).to(device)

    with torch.no_grad():
        risk_score = model(features_tensor).cpu().numpy()[0][0]

    # Decision logic
    if risk_score >= 0.7:
        decision = 'APPROVED'
        recommendation = 'Low risk. Approve with standard terms.'
    elif risk_score >= 0.4:
        decision = 'UNDER REVIEW'
        recommendation = 'Moderate risk. Additional verification needed.'
    else:
        decision = 'REJECTED'
        recommendation = 'High risk. Reject or require substantial collateral.'

    return {
        'risk_score': float(risk_score),
        'risk_percentage': float(risk_score * 100),
        'decision': decision,
        'recommendation': recommendation
    }

# Example usage
sample_application = {
    'income': 75000,
    'Credit_Score': 720,
    'loan_amount': 250000,
    'dtir1': 35.5,
    'LTV': 80.0,
    'age_numeric': 35,
    'open_credit': 5,
    'property_value': 312500,
    'rate_of_interest': 4.5,
    'loan_type': 'type1',
    'loan_purpose': 'p1',
    'Region': 'south'
}

result = predict_loan_application(model, sample_application, scaler, label_encoders, feature_columns, device)
print("\n" + "="*60)
print("LOAN APPLICATION ASSESSMENT")
print("="*60)
print(f"Risk Score: {result['risk_percentage']:.2f}%")
print(f"Decision: {result['decision']}")
print(f"Recommendation: {result['recommendation']}")
print("="*60)


In [ ]:
# Test on multiple applications
test_applications = [
    {'income': 120000, 'Credit_Score': 780, 'loan_amount': 300000, 'dtir1': 25, 'LTV': 70,
     'age': 42, 'open_credit': 8, 'property_value': 428571, 'rate_of_interest': 3.8,
     'loan_type': 'Conventional', 'loan_purpose': 'Home', 'Region': 'Urban'},

    {'income': 45000, 'Credit_Score': 620, 'loan_amount': 180000, 'dtir1': 42, 'LTV': 95,
     'age': 28, 'open_credit': 3, 'property_value': 189474, 'rate_of_interest': 5.5,
     'loan_type': 'FHA', 'loan_purpose': 'Home', 'Region': 'Rural'},

    {'income': 95000, 'Credit_Score': 700, 'loan_amount': 220000, 'dtir1': 32, 'LTV': 82,
     'age': 38, 'open_credit': 6, 'property_value': 268293, 'rate_of_interest': 4.2,
     'loan_type': 'Conventional', 'loan_purpose': 'Refinance', 'Region': 'Suburban'}
]

print("\n" + "="*70)
print("BATCH LOAN APPLICATION ASSESSMENTS")
print("="*70)

for idx, app in enumerate(test_applications, 1):
    result = predict_loan_application(app)
    print(f"\nApplication #{idx}:")
    print(f"  Income: ${app['income']:,} | Credit Score: {app['Credit_Score']} | Loan: ${app['loan_amount']:,}")
    print(f"  Risk Score: {result['risk_percentage']:.2f}% | Decision: {result['decision']}")
    print(f"  {result['recommendation']}")


# Summary

In [ ]:
# Generate comprehensive report
print("\n" + "="*70)
print("FINAL MODEL PERFORMANCE SUMMARY")
print("="*70)
print(f"\n📊 Dataset Statistics:")
print(f"   • Total samples: {len(df)}")
print(f"   • Training samples: {len(X_train)}")
print(f"   • Test samples: {len(X_test)}")
print(f"   • Number of features: {len(feature_columns)}")

print(f"\n🎯 Model Performance:")
print(f"   • Accuracy:  {accuracy*100:.2f}%")
print(f"   • Precision: {precision*100:.2f}%")
print(f"   • Recall:    {recall*100:.2f}%")
print(f"   • F1 Score:  {f1*100:.2f}%")
print(f"   • AUC Score: {roc_auc*100:.2f}%")

print(f"\n🏗️ Model Architecture:")
print(f"   • Input features: {input_dim}")
print(f"   • Hidden layers: 3 (64 → 32 → 16 neurons)")
print(f"   • Activation: ReLU with dropout regularization")
print(f"   • Output: Sigmoid (binary classification)")
print(f"   • Total parameters: {model.count_params():,}")

print(f"\n⚙️ Training Configuration:")
print(f"   • Optimizer: Adam")
print(f"   • Loss function: Binary Crossentropy")
print(f"   • Epochs trained: {len(history.history['loss'])}")
print(f"   • Batch size: {batch_size}")

print(f"\n✅ Model Status: Ready for deployment")
print("="*70)

# Export

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC'],
    'Score': [accuracy, precision, recall, f1, roc_auc],
    'Percentage': [f"{accuracy*100:.2f}%", f"{precision*100:.2f}%",
                   f"{recall*100:.2f}%", f"{f1*100:.2f}%", f"{roc_auc*100:.2f}%"]
})

# Save results to CSV
results_df.to_csv('model_evaluation_results.csv', index=False)
print("✅ Results saved to 'model_evaluation_results.csv'")

# Download results
files.download('model_evaluation_results.csv')
print("📥 Results downloaded")